In [1]:
import pandas as pd
import feature

In [2]:
df=pd.read_csv("../data/Amazon_Sale_Report.csv")


C:\Users\ABHIJITH\AppData\Local\Temp\ipykernel_2580\2237359375.py:1: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("../data/Amazon_Sale_Report.csv")


In [ ]:
df = df.drop(columns=['Unnamed: 22', 'fulfilled-by'])



In [5]:
df['promotion']=df['promotion'].fillna('No promotion')

In [6]:
multiple = ['ship-city', 'ship-state', 'ship-postal-code','currency']
df[multiple] = df[multiple].fillna('Unknown')

In [7]:
df.columns=df.columns.str.lower()
df.columns=df.columns.str.replace(' ','_')

In [8]:
df['ship-country']=df['ship-country'].fillna('IN')

In [9]:
df.loc[df['status'] == 'Cancelled', 'courier_status'] = 'Cancelled'
df.loc[df['status'] == 'Shipped - Delivered to Buyer', 'courier_status'] = 'Shipped'
df.loc[df['status'] == 'Shipped - Returned to Seller', 'courier_status'] = 'Shipped'

In [10]:
df.amount=df.amount.fillna(0)

In [11]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

C:\Users\ABHIJITH\AppData\Local\Temp\ipykernel_2580\2370506791.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'], errors='coerce')


In [12]:
df['date'] = df['date'].ffill()

In [13]:
df['month']=df['date'].dt.strftime('%B')

In [14]:
df['year']=df.date.dt.year

In [15]:
df['day_name']=df.date.dt.strftime('%A')

In [16]:
df['is_weekend']=df.date.dt.dayofweek.isin([5,6]).astype(int)

In [17]:
df['date'] = df['date'].dt.strftime('%Y-%m-%d')

In [18]:
sku_sales_velocity=df.groupby('sku')['qty'].transform('sum')


In [19]:
df['item_velocity']=pd.qcut(sku_sales_velocity,q=3,labels=['Slow Moving', 'Moderate', 'Fast Selling'])

In [20]:
category_counts = df.groupby('category')['qty'].sum().reset_index()
category_counts = category_counts.sort_values(by='qty', ascending=False).reset_index(drop=True)

category_counts['category_demand']=category_counts.index.map(feature.index_var)
category_dict=dict(zip(category_counts['category'],category_counts['category_demand']))


In [21]:
df['category_demand']=df.category.map(category_dict)

In [24]:
df.head()

,index,order_id,date,status,fulfilment,sales_channel_,ship-service-level,style,sku,category,...,ship-postal-code,ship-country,promotion,b2b,month,year,day_name,is_weekend,item_velocity,category_demand
0,0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,...,400081.0,IN,No promotion,False,April,2022,Saturday,1,Moderate,high demand
1,1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,...,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,April,2022,Saturday,1,Fast Selling,high demand
2,2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,...,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,April,2022,Saturday,1,Slow Moving,high demand
3,3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,...,605008.0,IN,No promotion,False,April,2022,Saturday,1,Fast Selling,high demand
4,4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,...,600073.0,IN,No promotion,False,April,2022,Saturday,1,Slow Moving,medium demand


In [76]:
%pip install sqlalchemy pymysql


Note: you may need to restart the kernel to use updated packages.


In [79]:

from sqlalchemy import create_engine
DB_USER = "root"       
DB_PASSWORD = "root"   
DB_HOST = "localhost"          
DB_PORT = "3306"                
DB_NAME = "Amazon_sales_report"  


connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(connection_string)


In [80]:
df.to_sql("amazon_sales_report_cleaned", con=engine, if_exists="replace", index=False)

128975